# TerraAlert - Landslide Susceptibility Model Training

## Mục tiêu
Huấn luyện mô hình **XGBoost / Random Forest** dự đoán mức độ nhạy cảm sạt lở đất (Landslide Susceptibility Mapping)

## Output
- `lsm_xgboost_model.pkl` - Model chính (XGBoost)
- `lsm_random_forest_model.pkl` - Model backup (Random Forest)
- `label_encoders.pkl` - Label encoders
- `model_metadata.json` - Metadata + metrics

## Yêu cầu
Upload file `landslide_dataset.csv` lên Kaggle Files trước khi chạy notebook này.

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, f1_score, accuracy_score
from sklearn.inspection import permutation_importance

import xgboost as xgb
import joblib
import json
from datetime import datetime
import os
import glob

print('Libraries loaded!')

## 1. Load Dataset

Dataset được tạo từ script `create_dataset.py` với các features:
- `latitude`, `longitude`: Tọa độ
- `elevation`: Độ cao (m)
- `slope`: Độ dốc (độ)
- `aspect`: Hướng dốc (0-360)
- `annual_precipitation`: Lượng mưa năm (mm)
- `max_daily_rainfall`: Mưa max ngày (mm)
- `rainfall_intensity`: Cường độ mưa TB (mm/day)
- `ndvi`: Chỉ số thực vật (-1 đến 1)
- `distance_to_fault`: Khoảng cách đứt gãy (km)
- `landslide`: Label (0: không sạt lở, 1: sạt lở)

In [ ]:
# Tìm file CSV
csv_candidates = [
    '/kaggle/input/landslide-dataset/landslide_dataset.csv',
    '/kaggle/input/landslide-dataset/*.csv',
    '/kaggle/working/landslide_dataset.csv',
    'landslide_dataset.csv',
    '/kaggle/input/*.csv',
]

raw_df = None
for pattern in csv_candidates:
    matches = glob.glob(pattern)
    if matches:
        csv_path = matches[0]
        raw_df = pd.read_csv(csv_path)
        print(f'Loaded: {csv_path}')
        break

if raw_df is None:
    print('ERROR: No CSV file found!')
    print('')
    print('Please upload landslide_dataset.csv to Kaggle Files:')
    print('  1. Run: python create_dataset.py --mode synthetic --output landslide_dataset.csv')
    print('  2. Upload landslide_dataset.csv to Kaggle Files (tab Files -> Upload)')
    print('  3. Re-run this notebook')
else:
    print(f'Shape: {raw_df.shape}')
    print(f'Columns: {raw_df.columns.tolist()}')
    print(f'\nFirst 5 rows:')
    display(raw_df.head())

In [ ]:
# Kiểm tra và làm sạch dữ liệu
if raw_df is not None:
    df = raw_df.copy()
    
    # Kiểm tra cột bắt buộc
    required_cols = ['latitude', 'longitude', 'landslide', 'elevation', 'slope', 'ndvi']
    missing = [c for c in required_cols if c not in df.columns]
    
    if missing:
        print(f'ERROR: Missing columns: {missing}')
        print(f'Available: {df.columns.tolist()}')
    else:
        # Fill missing values
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())
        
        # Thêm categorical features nếu chưa có
        if 'soil_type' not in df.columns:
            df['soil_type'] = np.random.choice(['clay', 'sand', 'silt', 'loam', 'rock'], len(df))
        if 'lithology' not in df.columns:
            df['lithology'] = np.random.choice(['sedimentary', 'igneous', 'metamorphic', 'alluvial'], len(df))
        if 'land_use' not in df.columns:
            df['land_use'] = np.random.choice(['forest', 'agriculture', 'urban', 'barren', 'grassland'], len(df))
        
        print(f'Dataset shape: {df.shape}')
        print(f'\nLandslide distribution:')
        print(df['landslide'].value_counts())
        print(f'\nFeature statistics:')
        display(df.describe().round(2))

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# Phân bố nhãn
if raw_df is not None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    df['landslide'].value_counts().plot(kind='bar', ax=axes[0], color=['#2ecc71', '#e74c3c'])
    axes[0].set_title('Phân bố nhãn Landslide')
    axes[0].set_xticklabels(['No Landslide (0)', 'Landslide (1)'], rotation=0)
    axes[0].set_ylabel('Count')
    
    df['landslide'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'])
    axes[1].set_title('Tỷ lệ Landslide')
    axes[1].set_ylabel('')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Correlation heatmap
if raw_df is not None:
    numerical_cols = ['elevation', 'slope', 'aspect', 'annual_precipitation', 
                      'max_daily_rainfall', 'ndvi', 'distance_to_fault', 'landslide']
    numerical_cols = [c for c in numerical_cols if c in df.columns]
    
    fig, ax = plt.subplots(figsize=(10, 8))
    corr_matrix = df[numerical_cols].corr()
    sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax)
    ax.set_title('Correlation Matrix')
    plt.tight_layout()
    plt.show()

In [ ]:
# Phân bố features theo landslide
if raw_df is not None:
    key_features = ['slope', 'annual_precipitation', 'elevation', 'distance_to_fault', 'ndvi']
    key_features = [f for f in key_features if f in df.columns]
    
    fig, axes = plt.subplots(1, len(key_features), figsize=(20, 4))
    for i, feat in enumerate(key_features):
        for label, color, name in [(0, '#2ecc71', 'No Landslide'), (1, '#e74c3c', 'Landslide')]:
            subset = df[df['landslide'] == label][feat]
            axes[i].hist(subset, bins=30, alpha=0.6, color=color, label=name, density=True)
        axes[i].set_title(feat)
        axes[i].legend(fontsize=8)
    
    plt.suptitle('Phân bố Features theo Landslide', y=1.02)
    plt.tight_layout()
    plt.show()

## 3. Feature Engineering

In [ ]:
if raw_df is not None:
    # Encode categorical features
    le_soil = LabelEncoder()
    le_litho = LabelEncoder()
    le_landuse = LabelEncoder()
    
    df['soil_encoded'] = le_soil.fit_transform(df['soil_type'])
    df['lithology_encoded'] = le_litho.fit_transform(df['lithology'])
    df['landuse_encoded'] = le_landuse.fit_transform(df['land_use'])
    
    # Feature Engineering
    df['slope_rainfall'] = df['slope'] * df['annual_precipitation'] / 1000
    df['elev_slope_ratio'] = df['elevation'] / (df['slope'] + 1)
    df['terrain_rugged'] = df['elevation'] * df['slope'] / 100
    df['rain_elev_idx'] = df['annual_precipitation'] * df['elevation'] / 10000
    df['fault_river'] = 1 / (df['distance_to_fault'] + 1)
    
    print('Feature engineering done!')
    print(f'Total features: {len(df.columns) - 3}')  # Trừ lat, lon, landslide

In [ ]:
if raw_df is not None:
    # Chọn features cho model
    feature_columns = [
        'elevation', 'slope', 'aspect',
        'annual_precipitation', 'max_daily_rainfall', 'rainfall_intensity',
        'distance_to_fault', 'soil_encoded', 'lithology_encoded',
        'landuse_encoded', 'ndvi',
        'slope_rainfall', 'elev_slope_ratio', 'terrain_rugged', 'rain_elev_idx', 'fault_river'
    ]
    
    # Chỉ giữ features có trong dataframe
    feature_columns = [f for f in feature_columns if f in df.columns]
    
    X = df[feature_columns].copy()
    y = df['landslide'].copy()
    
    # Fill missing
    X = X.fillna(X.median())
    
    print(f'Features: {feature_columns}')
    print(f'X shape: {X.shape}')
    print(f'y distribution: {y.value_counts().to_dict()}')

In [ ]:
if raw_df is not None:
    # Split train/test
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    
    print(f'Train: {X_train.shape}, Test: {X_test.shape}')
    print(f'Train positive rate: {y_train.mean():.2%}')
    print(f'Test positive rate: {y_test.mean():.2%}')

## 4. Train Models

In [ ]:
if raw_df is not None:
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    # Random Forest
    print('Training Random Forest...')
    rf = RandomForestClassifier(
        n_estimators=200, max_depth=15, random_state=42, n_jobs=-1, class_weight='balanced'
    )
    rf.fit(X_train, y_train)
    rf_cv = cross_val_score(rf, X_train, y_train, cv=cv, scoring='roc_auc')
    rf_pred = rf.predict(X_test)
    rf_proba = rf.predict_proba(X_test)[:, 1]
    print(f'  RF CV ROC-AUC: {rf_cv.mean():.4f} (+/- {rf_cv.std():.4f})')
    
    # XGBoost
    print('\nTraining XGBoost...')
    xgb_model = xgb.XGBClassifier(
        n_estimators=300, max_depth=8, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1,
        scale_pos_weight=len(y_train[y_train==0]) / len(y_train[y_train==1]),
        eval_metric='auc', use_label_encoder=False
    )
    xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
    xgb_cv = cross_val_score(xgb_model, X_train, y_train, cv=cv, scoring='roc_auc')
    xgb_pred = xgb_model.predict(X_test)
    xgb_proba = xgb_model.predict_proba(X_test)[:, 1]
    print(f'  XGB CV ROC-AUC: {xgb_cv.mean():.4f} (+/- {xgb_cv.std():.4f})')
    
    # So sánh
    print(f'\nTest ROC-AUC:')
    print(f'  Random Forest: {roc_auc_score(y_test, rf_proba):.4f}')
    print(f'  XGBoost:       {roc_auc_score(y_test, xgb_proba):.4f}')

## 5. Evaluation

In [ ]:
if raw_df is not None:
    # ROC Curves
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    for name, proba, color in [('Random Forest', rf_proba, '#3498db'), ('XGBoost', xgb_proba, '#e74c3c')]:
        fpr, tpr, _ = roc_curve(y_test, proba)
        auc = roc_auc_score(y_test, proba)
        axes[0].plot(fpr, tpr, color=color, label=f'{name} (AUC={auc:.3f})')
    
    axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5)
    axes[0].set_xlabel('False Positive Rate')
    axes[0].set_ylabel('True Positive Rate')
    axes[0].set_title('ROC Curves')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Confusion Matrix (XGBoost)
    cm = confusion_matrix(y_test, xgb_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
                xticklabels=['No Landslide', 'Landslide'],
                yticklabels=['No Landslide', 'Landslide'])
    axes[1].set_xlabel('Predicted')
    axes[1].set_ylabel('Actual')
    axes[1].set_title('Confusion Matrix (XGBoost)')
    
    plt.tight_layout()
    plt.show()
    
    print('\nClassification Report (XGBoost):')
    print(classification_report(y_test, xgb_pred, target_names=['No Landslide', 'Landslide']))

In [ ]:
if raw_df is not None:
    # Feature Importance
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    importance = xgb_model.feature_importances_
    feat_imp = pd.Series(importance, index=feature_columns).sort_values(ascending=True)
    feat_imp.tail(12).plot(kind='barh', ax=axes[0], color='#3498db')
    axes[0].set_title('XGBoost - Feature Importance (Built-in)')
    
    perm_imp = permutation_importance(xgb_model, X_test, y_test, n_repeats=10, random_state=42)
    perm_imp_series = pd.Series(perm_imp.importances_mean, index=feature_columns).sort_values(ascending=True)
    perm_imp_series.tail(12).plot(kind='barh', ax=axes[1], color='#e74c3c')
    axes[1].set_title('XGBoost - Permutation Importance')
    
    plt.tight_layout()
    plt.show()
    
    print('\nTop 10 Features:')
    for feat, imp in perm_imp_series.tail(10).items():
        print(f'  {feat:30s} {imp:.4f}')

## 6. Export Models

In [ ]:
if raw_df is not None:
    os.makedirs('output', exist_ok=True)
    
    metadata = {
        'model_name': 'TerraAlert_LSM',
        'version': '1.0.0',
        'trained_at': datetime.now().isoformat(),
        'features': feature_columns,
        'n_samples': len(X_train),
        'metrics': {
            'xgb_roc_auc': float(roc_auc_score(y_test, xgb_proba)),
            'xgb_f1': float(f1_score(y_test, xgb_pred)),
            'xgb_accuracy': float(accuracy_score(y_test, xgb_pred)),
            'rf_roc_auc': float(roc_auc_score(y_test, rf_proba)),
            'rf_f1': float(f1_score(y_test, rf_pred)),
        },
        'encoders': {
            'soil_type': le_soil.classes_.tolist(),
            'lithology': le_litho.classes_.tolist(),
            'land_use': le_landuse.classes_.tolist(),
        }
    }
    
    # Save
    joblib.dump(xgb_model, 'output/lsm_xgboost_model.pkl')
    joblib.dump(rf, 'output/lsm_random_forest_model.pkl')
    joblib.dump({'soil_type': le_soil, 'lithology': le_litho, 'land_use': le_landuse}, 'output/label_encoders.pkl')
    json.dump(metadata, open('output/model_metadata.json', 'w'), indent=2)
    
    print('Files saved to output/:')
    for f in os.listdir('output'):
        print(f'  {f}: {os.path.getsize(f"output/{f}")/1024:.0f} KB')
    
    print(f'\nModel Performance:')
    print(f'  XGBoost ROC-AUC: {metadata["metrics"]["xgb_roc_auc"]:.4f}')
    print(f'  Random Forest ROC-AUC: {metadata["metrics"]["rf_roc_auc"]:.4f}')
    
    print(f'\nDONE! Download files from output/ and copy to backend/app/slow_lane/ml/models/')